# FADNet: Solar Hotspot Detection
**Run cells in order. Cell 1 must run at the start of every Kaggle session.**

In [ ]:
# ==============================================================================
# CELL 1 — CoordAtt Patch (RUN FIRST, EVERY SESSION)
# ==============================================================================
!pip install -q ultralytics roboflow albumentations

import torch
import torch.nn as nn
import sys
import shutil
import pathlib

class h_sigmoid(nn.Module):
    def forward(self, x): return nn.functional.relu6(x + 3) / 6

class h_swish(nn.Module):
    def forward(self, x): return x * h_sigmoid()(x)

class CoordAtt(nn.Module):
    def __init__(self, inp, oup=None, reduction=32):
        super().__init__()
        oup = oup or inp
        mip = max(8, inp // reduction)
        self.conv1  = nn.Conv2d(inp, mip, 1, bias=False)
        self.bn1    = nn.BatchNorm2d(mip)
        self.act    = h_swish()
        self.conv_h = nn.Conv2d(mip, oup, 1, bias=False)
        self.conv_w = nn.Conv2d(mip, oup, 1, bias=False)

    def forward(self, x):
        B, C, H, W = x.shape
        x_h = x.mean(dim=3, keepdim=True)
        x_w = x.mean(dim=2, keepdim=True).permute(0,1,3,2)
        y   = torch.cat([x_h, x_w], dim=2)
        y   = self.act(self.bn1(self.conv1(y)))
        x_h, x_w = torch.split(y, [H, W], dim=2)
        x_w = x_w.permute(0,1,3,2)
        a_h = torch.sigmoid(self.conv_h(x_h))
        a_w = torch.sigmoid(self.conv_w(x_w))
        return x * a_h * a_w

def patch_ultralytics():
    import ultralytics.nn.modules as ult_modules
    import ultralytics.nn.tasks as tasks

    # Step 1: patch in-memory
    ult_modules.CoordAtt = CoordAtt
    ult_modules.coord_att = type(sys)('ultralytics.nn.modules.coord_att')
    ult_modules.coord_att.CoordAtt = CoordAtt
    ult_modules.coord_att.h_swish = h_swish
    ult_modules.coord_att.h_sigmoid = h_sigmoid
    sys.modules['ultralytics.nn.modules.coord_att'] = ult_modules.coord_att
    tasks.CoordAtt = CoordAtt

    # Step 2: write coord_att.py on-disk
    modules_dir = pathlib.Path(ult_modules.__file__).parent
    coord_att_path = modules_dir / 'coord_att.py'
    coord_att_path.write_text('''
import torch, torch.nn as nn

class h_sigmoid(nn.Module):
    def forward(self, x): return nn.functional.relu6(x + 3) / 6

class h_swish(nn.Module):
    def forward(self, x): return x * h_sigmoid()(x)

class CoordAtt(nn.Module):
    def __init__(self, inp, oup=None, reduction=32):
        super().__init__()
        oup = oup or inp
        mip = max(8, inp // reduction)
        self.conv1  = nn.Conv2d(inp, mip, 1, bias=False)
        self.bn1    = nn.BatchNorm2d(mip)
        self.act    = h_swish()
        self.conv_h = nn.Conv2d(mip, oup, 1, bias=False)
        self.conv_w = nn.Conv2d(mip, oup, 1, bias=False)
    def forward(self, x):
        B, C, H, W = x.shape
        x_h = x.mean(dim=3, keepdim=True)
        x_w = x.mean(dim=2, keepdim=True).permute(0,1,3,2)
        y   = torch.cat([x_h, x_w], dim=2)
        y   = self.act(self.bn1(self.conv1(y)))
        x_h, x_w = torch.split(y, [H, W], dim=2)
        x_w = x_w.permute(0,1,3,2)
        return x * torch.sigmoid(self.conv_h(x_h)) * torch.sigmoid(self.conv_w(x_w))
''')

    # Step 3: patch tasks.py (FORCE .py extension and inject at the very top)
    tasks_path = pathlib.Path(tasks.__file__).with_suffix('.py')
    tasks_text = tasks_path.read_text()
    if 'from ultralytics.nn.modules.coord_att import CoordAtt' not in tasks_text:
        tasks_path.write_text("from ultralytics.nn.modules.coord_att import CoordAtt\n" + tasks_text)

    # Step 4: Clear ALL Python caches so DDP multi-GPU workers are forced to read the raw .py files
    nn_dir = tasks_path.parent
    shutil.rmtree(nn_dir / '__pycache__', ignore_errors=True)
    shutil.rmtree(modules_dir / '__pycache__', ignore_errors=True)
    
    print('CoordAtt patched completely for multi-GPU DDP ✓')

patch_ultralytics()

In [ ]:
# ==============================================================================
# CELL 2 — Shims + Imports
# ==============================================================================
import numpy as np
np.trapz = np.trapezoid  # NumPy/Ultralytics version mismatch shim

import torch
import glob
import os
from ultralytics import YOLO

print('Imports done ✓')
print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
# ==============================================================================
# CELL 3 — Dataset Download + Label Remap (skip if already done)
# ==============================================================================
dataset_path = '/kaggle/working/Thermal-H&C-1'

if not os.path.exists(dataset_path):
    from roboflow import Roboflow
    from kaggle_secrets import UserSecretsClient

    user_secrets = UserSecretsClient()
    api_key = user_secrets.get_secret('ROBOFLOW_API_KEY')

    rf = Roboflow(api_key=api_key)
    project = rf.workspace('hotspotyolo').project('thermal-h-c')
    dataset = project.version(1).download('yolov11')
    print('Downloaded to:', dataset.location)

    # Remap: original 0=PV Hotspot, 1=Crack → corrected 0=Crack, 1=Hotspot
    for split in ['train', 'valid', 'test']:
        for lf in glob.glob(f'{dataset_path}/{split}/labels/*.txt'):
            with open(lf) as f:
                lines = f.readlines()
            new_lines = []
            for line in lines:
                parts = line.strip().split()
                parts[0] = '1' if parts[0] == '0' else '0'
                new_lines.append(' '.join(parts) + '\n')
            with open(lf, 'w') as f:
                f.writelines(new_lines)
    print('Labels remapped ✓')
else:
    print('Dataset already exists, skipping download.')

yaml_content = f"""
path: {dataset_path}
train: train/images
val: valid/images
test: test/images

nc: 2
names: ['Crack', 'Hotspot']
"""
with open('/kaggle/working/data_fixed.yaml', 'w') as f:
    f.write(yaml_content)
print('data_fixed.yaml written ✓')

In [ ]:
# ==============================================================================
# CELL 4 — Stage A Training (Frozen Backbone, 40 epochs)
# ==============================================================================
model_a = YOLO('/kaggle/input/datasets/vishokbadri/fadnet-aug/best.pt')

model_a.train(
    data='/kaggle/working/data_fixed.yaml',
    epochs=40,
    imgsz=640,
    batch=16,
    device='0,1',
    project='/kaggle/working/runs/detect',
    name='stageA_aug_v2',
    freeze=10,
    lr0=1e-3,
    lrf=0.01,
    warmup_epochs=3,
    warmup_momentum=0.8,
    momentum=0.937,
    mosaic=1.0,
    mixup=0.1,
    flipud=0.5,
    fliplr=0.5,
    degrees=15,
    translate=0.05,
    scale=0.1,
    weight_decay=0.0005,
    dropout=0.0,
    cls=1.5,
    patience=0,
    amp=True,
    cache=False,
    exist_ok=True,
    val=True,
)
print('Stage A done ✓')

In [ ]:
# ==============================================================================
# CELL 5 — Stage B Training (Full Fine-Tune, 40 epochs)
# ==============================================================================
model_b = YOLO('/kaggle/working/runs/detect/stageA_aug_v2/weights/best.pt')

model_b.train(
    data='/kaggle/working/data_fixed.yaml',
    epochs=40,
    imgsz=640,
    batch=16,
    device='0,1',
    project='/kaggle/working/runs/detect',
    name='stageB_aug_v2',
    lr0=1e-4,
    lrf=0.01,
    warmup_epochs=2,
    momentum=0.937,
    mosaic=1.0,
    mixup=0.1,
    flipud=0.5,
    fliplr=0.5,
    degrees=15,
    translate=0.05,
    scale=0.1,
    cls=1.5,
    weight_decay=0.0005,
    patience=0,
    amp=True,
    cache=False,
    exist_ok=True,
)
print('Stage B done ✓')

In [ ]:
# ==============================================================================
# CELL 6 — Stage C Part 1 (Mosaic ON, 30 epochs)
# ==============================================================================
model_c1 = YOLO('/kaggle/working/runs/detect/stageB_aug_v2/weights/best.pt')

model_c1.train(
    data='/kaggle/working/data_fixed.yaml',
    epochs=30,
    imgsz=640,
    batch=16,
    device='0,1',
    project='/kaggle/working/runs/detect',
    name='stageC_aug_v2_p1',
    lr0=5e-5,
    lrf=0.01,
    momentum=0.937,
    mosaic=1.0,
    mixup=0.05,
    flipud=0.5,
    fliplr=0.5,
    degrees=10,
    translate=0.05,
    scale=0.05,
    cls=1.5,
    weight_decay=0.0005,
    patience=0,
    amp=True,
    cache=False,
    exist_ok=True,
)
print('Stage C Part 1 done ✓')

In [ ]:
# ==============================================================================
# CELL 7 — Stage C Part 2 (Mosaic OFF, final 10 epochs)
# ==============================================================================
from ultralytics import YOLO
model_c2 = YOLO('/kaggle/working/runs/detect/stageC_aug_v2_p1/weights/best.pt')

# FIX: Forcefully purge the bad argument if it was inherited from the weights file
if 'grad_clip' in model_c2.overrides:
    del model_c2.overrides['grad_clip']

model_c2.train(
    data='/kaggle/working/data_fixed.yaml',
    epochs=10,
    imgsz=640,
    batch=16,
    device='0,1',
    project='/kaggle/working/runs/detect',
    name='stageC_aug_v2_p2',
    lr0=1e-5,
    lrf=0.01,
    momentum=0.937,
    mosaic=0.0,
    mixup=0.0,
    flipud=0.5,
    fliplr=0.5,
    degrees=5,
    translate=0.02,
    scale=0.02,
    cls=1.5,
    weight_decay=0.0005,
    patience=0,
    amp=True,
    cache=False,
    exist_ok=True,
)
print('Stage C done ✓')

In [ ]:
# CELL 8 — Final Evaluation
import pathlib
for cache in glob.glob('/kaggle/working/Thermal-H&C-1/**/*.cache', recursive=True):
    pathlib.Path(cache).unlink(missing_ok=True)

final_model = YOLO('/kaggle/working/runs/detect/stageC_aug_v2_p2/weights/best.pt')
metrics = final_model.val(
    data='/kaggle/working/data_fixed.yaml',
    conf=0.25,
    iou=0.35,
    split='test'
)
print('=' * 50)
print(f'mAP@0.5:      {metrics.box.map50:.4f}')
print(f'mAP@0.5:0.95: {metrics.box.map:.4f}')
print(f'Per-class:    {metrics.box.maps}')
print('=' * 50)

---
# FADNet — Inference & Model Diagnostics
**Cells 9–13: run after training is complete. No re-patch needed if in same session.**

In [ ]:
# ==============================================================================
# CELL 9 — Inference Config & Imports
# ==============================================================================
import numpy as np
np.trapz = np.trapezoid

import os, glob, time
import torch, cv2
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from pathlib import Path
from collections import defaultdict
from ultralytics import YOLO

# ── CONFIG — edit here only ───────────────────────────────────────────────────
MODEL_PATH  = '/kaggle/working/runs/detect/stageC_aug_v2_p2/weights/best.pt'
TEST_DIR    = '/kaggle/working/Thermal-H&C-1/test/images'
LABEL_DIR   = '/kaggle/working/Thermal-H&C-1/test/labels'
YAML_PATH   = '/kaggle/working/data_fixed.yaml'
OUTPUT_DIR  = '/kaggle/working/fadnet_inference_out'
CONF        = 0.05
IOU         = 0.35
CLASS_NAMES = ['Crack', 'Hotspot']
CLASS_COLORS_BGR = {0: (0, 0, 220), 1: (0, 140, 255)}  # Crack=red, Hotspot=orange
# ─────────────────────────────────────────────────────────────────────────────

os.makedirs(OUTPUT_DIR, exist_ok=True)
infer_model = YOLO(MODEL_PATH)
test_images = sorted(glob.glob(os.path.join(TEST_DIR, '*.*')))
print(f'Model loaded ✓  |  Test images: {len(test_images)}')
print(f'GPU: {torch.cuda.get_device_name(0)}')

In [ ]:
# ==============================================================================
# CELL 10 — Single Image Inference + Annotated Visualization
# ==============================================================================
def infer_single(image_path, conf=CONF, iou=IOU, save=True):
    img_bgr = cv2.imread(image_path)
    assert img_bgr is not None, f'Cannot read: {image_path}'
    H, W = img_bgr.shape[:2]

    t0 = time.perf_counter()
    results = infer_model.predict(image_path, conf=conf, iou=iou, verbose=False)[0]
    latency_ms = (time.perf_counter() - t0) * 1000

    annotated = img_bgr.copy()
    counts = defaultdict(int)

    for box in results.boxes:
        cls_id = int(box.cls.item())
        conf_v = box.conf.item()
        x1, y1, x2, y2 = map(int, box.xyxy[0].tolist())
        color = CLASS_COLORS_BGR[cls_id]
        label = f'{CLASS_NAMES[cls_id]} {conf_v:.2f}'
        counts[CLASS_NAMES[cls_id]] += 1
        cv2.rectangle(annotated, (x1, y1), (x2, y2), color, 2)
        (tw, th), _ = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, 0.55, 1)
        cv2.rectangle(annotated, (x1, y1 - th - 6), (x1 + tw + 4, y1), color, -1)
        cv2.putText(annotated, label, (x1+2, y1-4),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.55, (255,255,255), 1, cv2.LINE_AA)

    summary = [
        f'FADNet | {Path(image_path).name}',
        f'Latency: {latency_ms:.1f} ms',
        f'Crack:   {counts["Crack"]}',
        f'Hotspot: {counts["Hotspot"]}',
        f'Total:   {sum(counts.values())}',
    ]
    for i, line in enumerate(summary):
        y_pos = 22 + i * 22
        cv2.putText(annotated, line, (10, y_pos), cv2.FONT_HERSHEY_SIMPLEX, 0.55, (0,0,0), 3, cv2.LINE_AA)
        cv2.putText(annotated, line, (10, y_pos), cv2.FONT_HERSHEY_SIMPLEX, 0.55, (255,255,255), 1, cv2.LINE_AA)

    if save:
        out_path = os.path.join(OUTPUT_DIR, 'single_' + Path(image_path).name)
        cv2.imwrite(out_path, annotated)

    rgb = cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB)
    plt.figure(figsize=(10, 6))
    plt.imshow(rgb)
    plt.axis('off')
    plt.title(f'FADNet  |  Latency: {latency_ms:.1f}ms  |  Crack: {counts["Crack"]}  Hotspot: {counts["Hotspot"]}')
    plt.legend(handles=[
        mpatches.Patch(color=(220/255, 0, 0), label='Crack'),
        mpatches.Patch(color=(1, 140/255, 0), label='Hotspot'),
    ], loc='upper right', fontsize=9)
    plt.tight_layout()
    plt.show()
    print(f'Crack={counts["Crack"]}  Hotspot={counts["Hotspot"]}  Total={sum(counts.values())}  Latency={latency_ms:.1f}ms')
    return annotated, results

# Change index to inspect any specific image
annotated_img, raw_result = infer_single(test_images[0])

In [ ]:
# ==============================================================================
# CELL 13 — Confidence Distribution + Box Size + Detection Count Analysis
# ==============================================================================
conf_scores = defaultdict(list)
box_areas   = defaultdict(list)
boxes_per_image = defaultdict(list)

for img_path in test_images:
    result = infer_model.predict(img_path, conf=0.01, iou=IOU, verbose=False)[0]
    H, W   = result.orig_shape
    counts = defaultdict(int)
    for box in result.boxes:
        cls_id = int(box.cls.item())
        cv_val = box.conf.item()
        x1,y1,x2,y2 = box.xyxy[0].tolist()
        area = ((x2-x1)*(y2-y1)) / (H*W)
        name = CLASS_NAMES[cls_id]
        conf_scores[name].append(cv_val)
        box_areas[name].append(area)
        counts[name] += 1
    for name in CLASS_NAMES:
        boxes_per_image[name].append(counts[name])

colors = {'Crack': '#DC143C', 'Hotspot': '#FF8C00'}
fig, axes = plt.subplots(2, 2, figsize=(14, 9))
fig.suptitle('FADNet — Confidence & Detection Analysis', fontsize=13, fontweight='bold')

ax = axes[0,0]
for name in CLASS_NAMES:
    ax.hist(conf_scores[name], bins=30, alpha=0.6, label=name, color=colors[name], edgecolor='white')
ax.axvline(CONF, color='black', linestyle='--', linewidth=1.2, label=f'threshold={CONF}')
ax.set_xlabel('Confidence'); ax.set_ylabel('Count')
ax.set_title('Confidence Distribution (conf=0.01)'); ax.legend()

ax2 = axes[0,1]
for name in CLASS_NAMES:
    s = np.sort(conf_scores[name])
    ax2.plot(s, np.arange(1,len(s)+1)/len(s), label=name, color=colors[name], linewidth=2)
ax2.axvline(CONF, color='black', linestyle='--', linewidth=1.2, label=f'threshold={CONF}')
ax2.set_xlabel('Confidence'); ax2.set_ylabel('CDF')
ax2.set_title('Confidence CDF'); ax2.legend()

ax3 = axes[1,0]
for name in CLASS_NAMES:
    ax3.hist(np.array(box_areas[name])*100, bins=25, alpha=0.6,
             label=name, color=colors[name], edgecolor='white')
ax3.set_xlabel('Box Area (% of image)'); ax3.set_ylabel('Count')
ax3.set_title('Detected Box Size Distribution'); ax3.legend()

ax4 = axes[1,1]
bp = ax4.boxplot([boxes_per_image[n] for n in CLASS_NAMES],
                 labels=CLASS_NAMES, patch_artist=True,
                 medianprops=dict(color='black', linewidth=2))
for patch, name in zip(bp['boxes'], CLASS_NAMES):
    patch.set_facecolor(colors[name]); patch.set_alpha(0.6)
ax4.set_ylabel('Detections per Image'); ax4.set_title('Detection Count per Image')
for i, name in enumerate(CLASS_NAMES):
    vals = boxes_per_image[name]
    ax4.text(i+1, max(vals)+0.3, f'μ={np.mean(vals):.1f}\nmax={max(vals)}', ha='center', fontsize=9)

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'confidence_analysis.png'), dpi=150, bbox_inches='tight')
plt.show()

print('\n[Confidence Stats at conf=0.01]')
for name in CLASS_NAMES:
    c = np.array(conf_scores[name])
    sup = np.sum(c < CONF)
    print(f'  {name:<10} total={len(c)}  kept={np.sum(c>=CONF)}  '
          f'suppressed@{CONF}={sup}({100*sup/len(c):.1f}%)  '
          f'mean={c.mean():.3f}  median={np.median(c):.3f}  min={c.min():.3f}')

In [ ]:
# ==============================================================================
# CELL 14 — FP / FN Diagnosis: Worst Misses & False Alarms
# ==============================================================================
def load_gt(label_path, img_w, img_h):
    boxes = []
    if not os.path.exists(label_path): return boxes
    with open(label_path) as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) < 5: continue
            cls_id = int(parts[0])
            cx,cy,bw,bh = float(parts[1]),float(parts[2]),float(parts[3]),float(parts[4])
            x1=(cx-bw/2)*img_w; y1=(cy-bh/2)*img_h
            x2=(cx+bw/2)*img_w; y2=(cy+bh/2)*img_h
            boxes.append({'cls':cls_id,'box':[x1,y1,x2,y2]})
    return boxes

def iou_calc(b1, b2):
    xi1=max(b1[0],b2[0]); yi1=max(b1[1],b2[1])
    xi2=min(b1[2],b2[2]); yi2=min(b1[3],b2[3])
    inter=max(0,xi2-xi1)*max(0,yi2-yi1)
    return inter/((b1[2]-b1[0])*(b1[3]-b1[1])+(b2[2]-b2[0])*(b2[3]-b2[1])-inter+1e-9)

IOU_MATCH = 0.35
class_tp=defaultdict(int); class_fp=defaultdict(int); class_fn=defaultdict(int)
fp_images=[]; fn_images=[]

for img_path in test_images:
    stem = Path(img_path).stem
    img_bgr = cv2.imread(img_path)
    H, W = img_bgr.shape[:2]
    gt    = load_gt(os.path.join(LABEL_DIR, stem+'.txt'), W, H)
    result = infer_model.predict(img_path, conf=CONF, iou=IOU, verbose=False)[0]
    preds = [{'cls':int(b.cls.item()),'conf':b.conf.item(),'box':b.xyxy[0].tolist()}
             for b in result.boxes]

    matched_gt=set(); matched_pred=set()
    for pi, pred in enumerate(preds):
        best_iou, best_gi = 0, -1
        for gi, g in enumerate(gt):
            if g['cls']!=pred['cls'] or gi in matched_gt: continue
            v = iou_calc(pred['box'], g['box'])
            if v > best_iou: best_iou, best_gi = v, gi
        if best_iou >= IOU_MATCH:
            matched_gt.add(best_gi); matched_pred.add(pi)
            class_tp[CLASS_NAMES[pred['cls']]] += 1

    fp_list=[preds[i] for i in range(len(preds)) if i not in matched_pred]
    fn_list=[gt[i]    for i in range(len(gt))    if i not in matched_gt]
    for fp in fp_list: class_fp[CLASS_NAMES[fp['cls']]] += 1
    for fn in fn_list: class_fn[CLASS_NAMES[fn['cls']]] += 1
    if fp_list: fp_images.append((img_path, len(fp_list), fp_list))
    if fn_list: fn_images.append((img_path, len(fn_list), fn_list))

print('\n' + '='*60)
print('FP / FN / TP Breakdown')
print('='*60)
for name in CLASS_NAMES:
    tp=class_tp[name]; fp_c=class_fp[name]; fn_c=class_fn[name]
    prec=tp/(tp+fp_c+1e-9); rec=tp/(tp+fn_c+1e-9)
    print(f'  {name:<10}  TP={tp:>4}  FP={fp_c:>4}  FN={fn_c:>4}  Prec={prec:.3f}  Rec={rec:.3f}')
print(f'\n  Images with ≥1 FP: {len(fp_images)}/{len(test_images)}')
print(f'  Images with ≥1 FN: {len(fn_images)}/{len(test_images)}')
print('='*60)

def show_worst(cases, n, title, box_color, label_prefix):
    cases_sorted = sorted(cases, key=lambda x: x[1], reverse=True)
    n_show = min(n, len(cases_sorted))
    if n_show == 0: return
    fig, axes = plt.subplots(1, n_show, figsize=(5*n_show, 5))
    if n_show == 1: axes = [axes]
    fig.suptitle(title, fontsize=12, fontweight='bold')
    for ax, (img_path, count, boxes) in zip(axes, cases_sorted[:n_show]):
        img = cv2.cvtColor(cv2.imread(img_path), cv2.COLOR_BGR2RGB)
        for b in boxes:
            x1,y1,x2,y2 = map(int, b['box'])
            cv2.rectangle(img,(x1,y1),(x2,y2),box_color,2)
            lbl = f'{label_prefix}: {CLASS_NAMES[b["cls"]]}'+ (f' {b["conf"]:.2f}' if 'conf' in b else '')
            cv2.putText(img, lbl,(x1,max(y1-4,10)),cv2.FONT_HERSHEY_SIMPLEX,0.5,box_color,1)
        ax.imshow(img); ax.set_title(f'{Path(img_path).name}\ncount={count}', fontsize=9); ax.axis('off')
    plt.tight_layout()
    fname = title.replace(' ','_').replace('/','').lower()[:30] + '.png'
    plt.savefig(os.path.join(OUTPUT_DIR, fname), dpi=150, bbox_inches='tight')
    plt.show()

show_worst(fn_images, 4, 'Worst False Negatives (missed GT — GREEN)', (0,200,0), 'MISSED')
show_worst(fp_images, 4, 'Worst False Positives (wrong pred — BLUE)', (30,30,220), 'FP')
print('FP/FN plots saved.')

FADNet — Stage B (UNet Segmentation) + Stage C (YOLO Fine-tune)
Pipeline: Stage A YOLO (existing stageC_aug_v2_p2/best.pt) → Stage B UNet (EfficientNet-B4, weak bbox supervision) → Stage C YOLO (GT + UNet pseudo-labels, 30 epochs total)

Run Cell 1 first every session. Then run all cells in order.

Cell	Stage	Task	Epochs
4	B-prep	Pseudo-mask generation (bbox→pixel)	—
5-6	B-setup	SegDataset + UNet + loss config	—
7	B-train	EfficientNet-B4 UNet training	30
8	B-eval	Val dice + visual check	—
9	B-infer	Generate pseudo-labels for train split	—
10	C-prep	Merge GT + pseudo-labels, build dataset	—
11	C-p1	YOLO fine-tune, mosaic ON	20
12	C-p2	YOLO fine-tune, mosaic OFF (clean-up)	10
13	Eval	Before/After comparison table + chart	—
14-19	Inference	Batch inference, metrics, FP/FN	—


In [ ]:
# ==============================================================================
# CELL 15 — Installs + Imports
# ==============================================================================
import subprocess
subprocess.run(['pip', 'install', '-q', 'segmentation-models-pytorch'], check=True)

import numpy as np
np.trapz = np.trapezoid   # numpy/ultralytics shim

import os, glob, shutil, time
from pathlib import Path
from collections import defaultdict

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import cv2
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.colors import ListedColormap
from tqdm import tqdm

import albumentations as A
from albumentations.pytorch import ToTensorV2
import segmentation_models_pytorch as smp
from scipy import ndimage as sci_ndimage
from ultralytics import YOLO

CLASS_NAMES = ['Crack', 'Hotspot']
CLASS_COLORS = {'Crack': '#DC143C', 'Hotspot': '#FF8C00'}

print('All imports done ✓')
print('GPU:', torch.cuda.get_device_name(0))


In [ ]:
# ==============================================================================
# CELL 16 — Generate Pseudo-GT Masks from YOLO Bounding Box Labels
# Converts each YOLO bbox annotation to a filled 2D mask.
# Pixel classes: 0=background, 1=Crack, 2=Hotspot
# ==============================================================================
MASK_DIR = '/kaggle/working/pseudo_masks'
SPLIT_DIRS = {
    'train': ('/kaggle/working/Thermal-H&C-1/train/images',
              '/kaggle/working/Thermal-H&C-1/train/labels'),
    'valid': ('/kaggle/working/Thermal-H&C-1/valid/images',
              '/kaggle/working/Thermal-H&C-1/valid/labels'),
    'test':  ('/kaggle/working/Thermal-H&C-1/test/images',
              '/kaggle/working/Thermal-H&C-1/test/labels'),
}

def yolo_to_mask(label_path, img_h, img_w):
    # pixel values: 0=bg, 1=Crack, 2=Hotspot
    mask = np.zeros((img_h, img_w), dtype=np.uint8)
    if not os.path.exists(label_path):
        return mask
    with open(label_path) as f:
        for line in f:
            p = line.strip().split()
            if len(p) < 5: continue
            cls_id = int(p[0])
            cx, cy, bw, bh = map(float, p[1:5])
            x1 = max(0, int((cx - bw/2) * img_w))
            y1 = max(0, int((cy - bh/2) * img_h))
            x2 = min(img_w, int((cx + bw/2) * img_w))
            y2 = min(img_h, int((cy + bh/2) * img_h))
            mask[y1:y2, x1:x2] = cls_id + 1  # 0->1 Crack, 1->2 Hotspot
    return mask

total = 0
for split, (img_dir, lbl_dir) in SPLIT_DIRS.items():
    out_dir = os.path.join(MASK_DIR, split)
    os.makedirs(out_dir, exist_ok=True)
    for img_path in sorted(glob.glob(os.path.join(img_dir, '*.*'))):
        stem = Path(img_path).stem
        img  = cv2.imread(img_path)
        if img is None: continue
        H, W = img.shape[:2]
        mask = yolo_to_mask(os.path.join(lbl_dir, stem + '.txt'), H, W)
        np.save(os.path.join(out_dir, stem + '.npy'), mask)
        total += 1
print(f'Generated {total} pseudo-masks -> {MASK_DIR}')

# Visualize 4 train samples
sample_imgs = sorted(glob.glob('/kaggle/working/Thermal-H&C-1/train/images/*.*'))[:4]
cmap3 = ListedColormap(['#111111', '#DC143C', '#FF8C00'])
fig, axes = plt.subplots(2, 4, figsize=(18, 8))
fig.suptitle('Pseudo-GT Masks (filled bboxes)  |  Red=Crack  Orange=Hotspot',
             fontsize=12, fontweight='bold')
for i, ip in enumerate(sample_imgs):
    stem = Path(ip).stem
    img_rgb = cv2.cvtColor(cv2.imread(ip), cv2.COLOR_BGR2RGB)
    mask    = np.load(f'/kaggle/working/pseudo_masks/train/{stem}.npy')
    axes[0,i].imshow(img_rgb); axes[0,i].axis('off'); axes[0,i].set_title(stem[:18])
    axes[1,i].imshow(mask, cmap=cmap3, vmin=0, vmax=2); axes[1,i].axis('off')
plt.tight_layout()
plt.savefig('/kaggle/working/pseudo_mask_samples.png', dpi=120, bbox_inches='tight')
plt.show()


In [ ]:
# ==============================================================================
# CELL 17 — SegDataset + DataLoaders (EfficientNet-B4 UNet input: 512x512)
# ==============================================================================
IMG_SIZE = 512

train_tfm = A.Compose([
    A.Resize(IMG_SIZE, IMG_SIZE),
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.5),
    A.RandomRotate90(p=0.5),
    A.RandomBrightnessContrast(brightness_limit=0.2, contrast_limit=0.2, p=0.4),
    A.GaussNoise(var_limit=(10, 40), p=0.2),
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2(),
])
val_tfm = A.Compose([
    A.Resize(IMG_SIZE, IMG_SIZE),
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2(),
])

class SegDataset(Dataset):
    def __init__(self, img_dir, mask_dir, transform=None):
        self.img_paths = sorted(glob.glob(os.path.join(img_dir, '*.*')))
        self.mask_dir  = mask_dir
        self.transform = transform

    def __len__(self): return len(self.img_paths)

    def __getitem__(self, idx):
        ip    = self.img_paths[idx]
        stem  = Path(ip).stem
        img   = cv2.cvtColor(cv2.imread(ip), cv2.COLOR_BGR2RGB)
        mask  = np.load(os.path.join(self.mask_dir, stem + '.npy')).astype(np.uint8)
        if self.transform:
            aug   = self.transform(image=img, mask=mask)
            img_t = aug['image']
            msk_t = aug['mask'].long()
        else:
            img_t = torch.from_numpy(img).permute(2,0,1).float() / 255.
            msk_t = torch.from_numpy(mask).long()
        return img_t, msk_t

train_ds = SegDataset('/kaggle/working/Thermal-H&C-1/train/images',
                      '/kaggle/working/pseudo_masks/train', train_tfm)
val_ds   = SegDataset('/kaggle/working/Thermal-H&C-1/valid/images',
                      '/kaggle/working/pseudo_masks/valid', val_tfm)

train_loader = DataLoader(train_ds, batch_size=8, shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=8, shuffle=False, num_workers=2, pin_memory=True)

print(f'Train: {len(train_ds)} imgs  |  Val: {len(val_ds)} imgs')
print(f'Train batches: {len(train_loader)}  |  Val batches: {len(val_loader)}')


In [ ]:
# ==============================================================================
# CELL 18 — VRAM Flush (delete all YOLO handles before UNet loads)
# ==============================================================================
import gc, torch

for _var in ['model_a', 'model_b', 'model_c1', 'model_c2',
             'final_model', 'infer_model']:
    if _var in dir():
        del globals()[_var]

gc.collect()
torch.cuda.empty_cache()
print(f'VRAM after flush: {torch.cuda.memory_allocated()/1e9:.2f} GB allocated')
print(f'VRAM reserved:    {torch.cuda.memory_reserved()/1e9:.2f} GB reserved')


In [ ]:
# ==============================================================================
# CELL 19 — UNet Model + Loss + Optimizer
# EfficientNet-B4 encoder (matches HOTSPOT-YOLO backbone), 3-class output
# ==============================================================================
NUM_CLASSES = 3   # 0=bg, 1=Crack, 2=Hotspot

unet = smp.Unet(
    encoder_name='efficientnet-b4',
    encoder_weights='imagenet',
    in_channels=3,
    classes=NUM_CLASSES,
    activation=None,        # raw logits, softmax applied in loss
).cuda()

total_p   = sum(p.numel() for p in unet.parameters())
encoder_p = sum(p.numel() for p in unet.encoder.parameters())
print(f'UNet total params : {total_p:,}')
print(f'  encoder (EffNet-B4) : {encoder_p:,}')
print(f'  decoder + head      : {total_p - encoder_p:,}')

# Dice loss on foreground classes + class-weighted CE to suppress bg dominance
dice_loss = smp.losses.DiceLoss(mode='multiclass', classes=[1, 2], from_logits=True)
ce_loss   = nn.CrossEntropyLoss(weight=torch.tensor([0.1, 1.5, 1.5]).cuda())

def combined_loss(logits, masks):
    return dice_loss(logits, masks) + ce_loss(logits, masks)

# Encoder LR 30x lower than decoder (warm up encoder gently)
optimizer = torch.optim.AdamW([
    {'params': unet.encoder.parameters(),          'lr': 1e-5},
    {'params': unet.decoder.parameters(),          'lr': 3e-4},
    {'params': unet.segmentation_head.parameters(),'lr': 3e-4},
], weight_decay=1e-4)

UNET_EPOCHS = 30
scheduler   = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=UNET_EPOCHS, eta_min=1e-6)
scaler      = torch.cuda.amp.GradScaler()

print('Loss / optimizer / scheduler ready ✓')


In [ ]:
# ==============================================================================
# CELL 20 — Train UNet — Stage B (30 epochs, AMP, gradient clipping)
# ==============================================================================
UNET_CKPT    = '/kaggle/working/unet_stageb_best.pth'
UNFREEZE_EP  = 10   # bump encoder LR after this epoch
best_val_dice = 0.0
history = {'train_loss': [], 'val_loss': [], 'val_dice': []}

def mean_dice(preds, targets):
    scores = []
    for c in [1, 2]:
        p = (preds == c).float(); t = (targets == c).float()
        inter = (p * t).sum(); union = p.sum() + t.sum()
        scores.append((2 * inter + 1e-6) / (union + 1e-6))
    return torch.stack(scores).mean().item()

for ep in range(1, UNET_EPOCHS + 1):

    if ep == UNFREEZE_EP + 1:
        optimizer.param_groups[0]['lr'] = 5e-5
        print(f'  [Ep {ep}] Encoder LR raised to 5e-5 ✓')

    # Train
    unet.train(); tr_loss = 0.0
    for imgs, masks in tqdm(train_loader, desc=f'Ep {ep:02d}/{UNET_EPOCHS} train', leave=False):
        imgs, masks = imgs.cuda(), masks.cuda()
        optimizer.zero_grad()
        with torch.cuda.amp.autocast():
            loss = combined_loss(unet(imgs), masks)
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        nn.utils.clip_grad_norm_(unet.parameters(), 1.0)
        scaler.step(optimizer); scaler.update()
        tr_loss += loss.item()
    scheduler.step()

    # Validate
    unet.eval(); vl_loss = 0.0; all_p = []; all_t = []
    with torch.no_grad():
        for imgs, masks in val_loader:
            imgs, masks = imgs.cuda(), masks.cuda()
            with torch.cuda.amp.autocast():
                logits = unet(imgs)
                vl_loss += combined_loss(logits, masks).item()
            all_p.append(logits.argmax(dim=1).cpu())
            all_t.append(masks.cpu())
    vdice = mean_dice(torch.cat(all_p), torch.cat(all_t))

    history['train_loss'].append(tr_loss / len(train_loader))
    history['val_loss'].append(vl_loss / len(val_loader))
    history['val_dice'].append(vdice)

    tag = ''
    if vdice > best_val_dice:
        best_val_dice = vdice
        torch.save(unet.state_dict(), UNET_CKPT)
        tag = '  * best'

    print(f'Ep {ep:02d}/{UNET_EPOCHS}  '
          f'tr_loss={tr_loss/len(train_loader):.4f}  '
          f'vl_loss={vl_loss/len(val_loader):.4f}  '
          f'vl_dice={vdice:.4f}  '
          f'lr={scheduler.get_last_lr()[0]:.1e}{tag}')

# Plot curves
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))
fig.suptitle('Stage B UNet — Training Curves', fontsize=12, fontweight='bold')
ax1.plot(history['train_loss'], label='Train', color='#DC143C')
ax1.plot(history['val_loss'],   label='Val',   color='#FF8C00')
ax1.set_xlabel('Epoch'); ax1.set_ylabel('Loss'); ax1.legend(); ax1.set_title('Combined Loss')
ax2.plot(history['val_dice'], color='#228B22', linewidth=2)
ax2.set_xlabel('Epoch'); ax2.set_ylabel('Dice')
ax2.set_title(f'Val Dice  (best = {best_val_dice:.4f})')
plt.tight_layout()
plt.savefig('/kaggle/working/unet_curves.png', dpi=120, bbox_inches='tight')
plt.show()
print(f'Stage B complete.  Best val Dice = {best_val_dice:.4f}')


In [ ]:
# ==============================================================================
# CELL 21 — UNet Validation: Per-Class Dice + Visual Comparison
# ==============================================================================
unet.load_state_dict(torch.load(UNET_CKPT))
unet.eval()

class_dice_scores = defaultdict(list)
with torch.no_grad():
    for imgs, masks in val_loader:
        imgs, masks = imgs.cuda(), masks.cuda()
        preds = unet(imgs).argmax(dim=1)
        for b in range(preds.shape[0]):
            for c_idx, name in enumerate(CLASS_NAMES):
                cls_id = c_idx + 1
                p = (preds[b] == cls_id).float()
                t = (masks[b] == cls_id).float()
                if t.sum() == 0 and p.sum() == 0: continue
                inter = (p * t).sum()
                class_dice_scores[name].append(
                    (2*inter / (p.sum() + t.sum() + 1e-6)).item()
                )

print('\n' + '='*50)
print('Stage B UNet — Per-Class Dice (val split)')
print('='*50)
for name in CLASS_NAMES:
    sc = class_dice_scores[name]
    print(f'  {name:<10}  mean={np.mean(sc):.4f}  median={np.median(sc):.4f}  n={len(sc)}')
print('='*50)

# Visual comparison on 3 val images
val_sample = sorted(glob.glob('/kaggle/working/Thermal-H&C-1/valid/images/*.*'))[:3]
cmap3 = ListedColormap(['#111111', '#DC143C', '#FF8C00'])
fig, axes = plt.subplots(3, 3, figsize=(14, 11))
fig.suptitle('UNet Predictions vs GT  (val set)', fontsize=12, fontweight='bold')
for ax, t in zip(axes[0], ['Input Image', 'GT Mask', 'UNet Prediction']):
    ax.set_title(t, fontsize=10, fontweight='bold')
for row, ip in enumerate(val_sample):
    stem    = Path(ip).stem
    img_rgb = cv2.cvtColor(cv2.imread(ip), cv2.COLOR_BGR2RGB)
    gt_mask = np.load(f'/kaggle/working/pseudo_masks/valid/{stem}.npy')
    aug     = val_tfm(image=img_rgb, mask=gt_mask)
    inp     = aug['image'].unsqueeze(0).cuda()
    with torch.no_grad():
        pred = unet(inp).argmax(dim=1)[0].cpu().numpy()
    axes[row,0].imshow(img_rgb);  axes[row,0].axis('off')
    axes[row,1].imshow(gt_mask, cmap=cmap3, vmin=0, vmax=2); axes[row,1].axis('off')
    axes[row,2].imshow(pred,    cmap=cmap3, vmin=0, vmax=2); axes[row,2].axis('off')
handles = [mpatches.Patch(color='#DC143C', label='Crack'),
           mpatches.Patch(color='#FF8C00', label='Hotspot')]
fig.legend(handles=handles, loc='lower center', ncol=2, fontsize=10)
plt.tight_layout()
plt.savefig('/kaggle/working/unet_val_preds.png', dpi=120, bbox_inches='tight')
plt.show()


In [ ]:
# ==============================================================================
# CELL 22 — Generate UNet Pseudo-Labels for Train Split
# Runs UNet inference on every training image, extracts connected components
# per class from softmax output, converts to YOLO bbox format.
# ==============================================================================
PSEUDO_LBL_DIR = '/kaggle/working/pseudo_labels_unet/train'
os.makedirs(PSEUDO_LBL_DIR, exist_ok=True)

CONF_THRESH = 0.45   # softmax probability threshold per pixel
MIN_AREA_PX = 64     # minimum component area (pixels at 512x512)

unet.load_state_dict(torch.load(UNET_CKPT))
unet.eval()

train_img_paths = sorted(glob.glob('/kaggle/working/Thermal-H&C-1/train/images/*.*'))
generated = 0

for img_path in tqdm(train_img_paths, desc='Generating pseudo-labels'):
    stem    = Path(img_path).stem
    img_bgr = cv2.imread(img_path)
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)

    aug = val_tfm(image=img_rgb, mask=np.zeros(img_rgb.shape[:2], np.uint8))
    inp = aug['image'].unsqueeze(0).cuda()

    with torch.no_grad():
        probs = torch.softmax(unet(inp), dim=1)[0].cpu().numpy()  # (3, 512, 512)

    lines = []
    for cls_id in [0, 1]:                   # YOLO cls: 0=Crack, 1=Hotspot
        prob_map = probs[cls_id + 1]         # UNet channel: 1=Crack, 2=Hotspot
        binary   = (prob_map > CONF_THRESH).astype(np.uint8)
        labeled, n_comp = sci_ndimage.label(binary)

        for lv in range(1, n_comp + 1):
            comp = (labeled == lv)
            if comp.sum() < MIN_AREA_PX: continue
            rows_idx = np.where(comp.any(axis=1))[0]
            cols_idx = np.where(comp.any(axis=0))[0]
            y1, y2   = rows_idx[0], rows_idx[-1]
            x1, x2   = cols_idx[0], cols_idx[-1]
            cx = (x1 + x2) / 2 / IMG_SIZE
            cy = (y1 + y2) / 2 / IMG_SIZE
            bw = (x2 - x1)     / IMG_SIZE
            bh = (y2 - y1)     / IMG_SIZE
            if bw > 0.01 and bh > 0.01:
                lines.append(f'{cls_id} {cx:.6f} {cy:.6f} {bw:.6f} {bh:.6f}')

    with open(os.path.join(PSEUDO_LBL_DIR, stem + '.txt'), 'w') as f:
        f.write('\n'.join(lines))
    if lines: generated += 1

print(f'Images with UNet detections: {generated}/{len(train_img_paths)}')
print(f'Pseudo-labels saved -> {PSEUDO_LBL_DIR}')


Stage C — YOLO Fine-tune on Merged GT + UNet Labels
Builds a symlinked dataset with original images + merged (GT + pseudo) labels, then continues from the existing stageC_aug_v2_p2 best.pt.

In [ ]:
# ==============================================================================
# CELL 23 — Merge GT + UNet Pseudo-Labels + Build Stage C Dataset
# GT boxes are always kept. Pseudo boxes are added only if they don't overlap
# any GT box at IoU >= 0.5 (avoids double-labelling existing annotations).
# ==============================================================================
GT_LBL_DIR     = '/kaggle/working/Thermal-H&C-1/train/labels'
MERGED_LBL_DIR = '/kaggle/working/merged_labels/train'
STAGEC_DATA    = '/kaggle/working/stagec_dataset'
os.makedirs(MERGED_LBL_DIR, exist_ok=True)

def load_boxes(path):
    if not os.path.exists(path): return []
    with open(path) as f:
        return [ln.strip() for ln in f if ln.strip()]

def box_iou_yolo(a, b):
    # a, b are tuples (cls, cx, cy, bw, bh) normalized
    if a[0] != b[0]: return 0.0
    ax1, ay1 = a[1]-a[3]/2, a[2]-a[4]/2
    ax2, ay2 = a[1]+a[3]/2, a[2]+a[4]/2
    bx1, by1 = b[1]-b[3]/2, b[2]-b[4]/2
    bx2, by2 = b[1]+b[3]/2, b[2]+b[4]/2
    ix1, iy1 = max(ax1,bx1), max(ay1,by1)
    ix2, iy2 = min(ax2,bx2), min(ay2,by2)
    inter = max(0,ix2-ix1)*max(0,iy2-iy1)
    ua = (ax2-ax1)*(ay2-ay1); ub = (bx2-bx1)*(by2-by1)
    return inter / (ua + ub - inter + 1e-9)

def merge_labels(gt_lines, pseudo_lines, iou_thresh=0.5):
    def parse(lines):
        out = []
        for ln in lines:
            p = ln.split()
            out.append((int(p[0]), *map(float, p[1:5])))
        return out
    gt_p = parse(gt_lines); ps_p = parse(pseudo_lines)
    kept = [pb for pb in ps_p if all(box_iou_yolo(pb, gb) < iou_thresh for gb in gt_p)]
    final = gt_p + kept
    return [f'{b[0]} {b[1]:.6f} {b[2]:.6f} {b[3]:.6f} {b[4]:.6f}' for b in final]

net_added = 0
for ip in sorted(glob.glob('/kaggle/working/Thermal-H&C-1/train/images/*.*')):
    stem  = Path(ip).stem
    gt_b  = load_boxes(os.path.join(GT_LBL_DIR,     stem + '.txt'))
    ps_b  = load_boxes(os.path.join(PSEUDO_LBL_DIR, stem + '.txt'))
    merged = merge_labels(gt_b, ps_b)
    net_added += len(merged) - len(gt_b)
    with open(os.path.join(MERGED_LBL_DIR, stem + '.txt'), 'w') as f:
        f.write('\n'.join(merged))

print(f'Net pseudo-boxes added after NMS dedup: +{net_added}')

# Symlinked dataset so YOLO finds labels under train/labels/
os.makedirs(f'{STAGEC_DATA}/train/images', exist_ok=True)
os.makedirs(f'{STAGEC_DATA}/train/labels', exist_ok=True)

for ip in sorted(glob.glob('/kaggle/working/Thermal-H&C-1/train/images/*.*')):
    dst = f'{STAGEC_DATA}/train/images/{Path(ip).name}'
    if not os.path.exists(dst): os.symlink(ip, dst)

for lp in sorted(glob.glob(f'{MERGED_LBL_DIR}/*.txt')):
    shutil.copy2(lp, f'{STAGEC_DATA}/train/labels/')

for split in ['valid', 'test']:
    src = f'/kaggle/working/Thermal-H&C-1/{split}'
    dst = f'{STAGEC_DATA}/{split}'
    if not os.path.exists(dst): os.symlink(src, dst)

yaml_stageC = f"""path: {STAGEC_DATA}
train: train/images
val: valid/images
test: test/images
nc: 2
names: ['Crack', 'Hotspot']
"""
with open('/kaggle/working/data_stageC_unet.yaml', 'w') as f:
    f.write(yaml_stageC)

print(f'Stage C dataset: '
      f'{len(glob.glob(STAGEC_DATA+"/train/images/*.*"))} images, '
      f'{len(glob.glob(STAGEC_DATA+"/train/labels/*.txt"))} labels')
print('data_stageC_unet.yaml ready ✓')


In [ ]:
# ==============================================================================
# CELL 24 — Stage C Part 1: Mosaic ON, 30 epochs  [UPDATED]
# Fine-tunes from stageC_aug_v2_p2/best.pt on pseudo-label dataset
#
# Changes vs previous version:
#   • Fixed FileNotFoundError: merged both cache-clear globs into one safe loop
#     using pathlib.unlink(missing_ok=True) so already-deleted files don't crash
#   • Removed grad_clip=1.0 (not a valid Ultralytics train arg; causes override
#     inheritance issues when loading weights in the next cell)
#   • Epochs raised from 20 → 30 to match the old run that hit 90.8%
# ==============================================================================
import glob
import os
import pathlib
from ultralytics import YOLO

STAGEC_DATA = '/kaggle/working/stagec_dataset'

# Safe cache clear — won't crash if a file was already deleted by the first pass
for pattern in ['/kaggle/working/**/*.cache', f'{STAGEC_DATA}/**/*.cache']:
    for cache in glob.glob(pattern, recursive=True):
        pathlib.Path(cache).unlink(missing_ok=True)

model_c1 = YOLO('/kaggle/working/runs/detect/stageC_aug_v2_p2/weights/best.pt')

# Purge any bad inherited overrides from the loaded weights
for bad_key in ['grad_clip']:
    model_c1.overrides.pop(bad_key, None)

model_c1.train(
    data='/kaggle/working/data_stageC_unet.yaml',
    epochs=30,                  # was 20 — raised to 30 to match old run
    imgsz=640,
    batch=16,
    device='0,1',
    project='/kaggle/working/runs/detect',
    name='stageC_unet_p1',
    lr0=5e-5,
    lrf=0.01,
    momentum=0.937,
    mosaic=1.0,
    mixup=0.05,
    flipud=0.5,
    fliplr=0.5,
    degrees=10,
    translate=0.05,
    scale=0.05,
    cls=1.5,
    weight_decay=0.0005,
    # grad_clip removed — was causing override issues
    patience=0,
    amp=True,
    cache=False,
    exist_ok=True,
    val=True,
)
print('Stage C Part 1 done ✓')


In [ ]:

# ==============================================================================
# CELL 25 — Stage C Part 2: Mosaic OFF, 10 epochs — clean-up pass  [UPDATED]
#
# Changes vs previous version:
#   • Switched data from data_stageC_unet.yaml → data_fixed.yaml (clean GT only)
#     This is the critical fix: the mosaic-off pass should recalibrate the model
#     on REAL, noise-free annotations — not pseudo-labels. Ending on clean data
#     is what pushed the old run to 90.8% on the test split.
#   • Removed grad_clip=1.0
#   • Added override purge before loading weights (same as old Cell 7 fix)
# ==============================================================================
from ultralytics import YOLO

model_c2 = YOLO('/kaggle/working/runs/detect/stageC_unet_p1/weights/best.pt')

# Purge any bad inherited overrides from the loaded weights
for bad_key in ['grad_clip']:
    model_c2.overrides.pop(bad_key, None)

model_c2.train(
    data='/kaggle/working/data_fixed.yaml',   # ← KEY CHANGE: clean GT, not pseudo-labels
    epochs=10,
    imgsz=640,
    batch=16,
    device='0,1',
    project='/kaggle/working/runs/detect',
    name='stageC_unet_p2',
    lr0=1e-5,
    lrf=0.01,
    momentum=0.937,
    mosaic=0.0,               # OFF — this is the domain-shift fix
    mixup=0.0,                # OFF
    flipud=0.5,
    fliplr=0.5,
    degrees=5,
    translate=0.02,
    scale=0.02,
    cls=1.5,
    weight_decay=0.0005,
    # grad_clip removed
    patience=0,
    amp=True,
    cache=False,
    exist_ok=True,
)
print('Stage C Part 2 done ✓')

In [ ]:
# ==============================================================================
# CELL 26 — Final Evaluation: Before vs After UNet Enrichment
# ==============================================================================
import pathlib

# Safe cache clear — won't crash if already deleted by the other DDP worker
for cache in glob.glob('/kaggle/working/**/*.cache', recursive=True):
    pathlib.Path(cache).unlink(missing_ok=True)

# Before: original FADNet (no UNet stage)
orig_model   = YOLO('/kaggle/working/runs/detect/stageC_aug_v2_p2/weights/best.pt')
orig_metrics = orig_model.val(data='/kaggle/working/data_fixed.yaml',
                               conf=0.25, iou=0.35, split='test')

# After: UNet-enriched Stage C
new_model   = YOLO('/kaggle/working/runs/detect/stageC_unet_p2/weights/best.pt')
new_metrics = new_model.val(data='/kaggle/working/data_fixed.yaml',
                             conf=0.25, iou=0.35, split='test')

def fmt(m):
    return (m.box.map50, m.box.map, m.box.mp, m.box.mr,
            m.box.ap50[0], m.box.ap50[1])

oa, ob  = fmt(orig_metrics), fmt(new_metrics)
labels  = ['mAP@0.5', 'mAP@0.5:0.95', 'Precision', 'Recall', 'AP50-Crack', 'AP50-Hotspot']

print('\n' + '='*68)
print(f'  {"Metric":<18}  {"Before":>16}  {"After (+UNet)":>16}  {"Delta":>8}')
print('-'*68)
for lbl, bef, aft in zip(labels, oa, ob):
    delta = aft - bef
    sign  = '+' if delta >= 0 else ''
    print(f'  {lbl:<18}  {bef:>16.4f}  {aft:>16.4f}  {sign}{delta:>7.4f}')
print('='*68)
print(f'\nFinal model -> /kaggle/working/runs/detect/stageC_unet_p2/weights/best.pt')

# Bar chart comparison
x = np.arange(len(labels))
fig, ax = plt.subplots(figsize=(13, 5))
fig.suptitle('FADNet — Before vs After UNet Stage B Enrichment', fontsize=13, fontweight='bold')
b1 = ax.bar(x - 0.18, oa, 0.33, label='Before (YOLO-only)', color='#4682B4', alpha=0.85)
b2 = ax.bar(x + 0.18, ob, 0.33, label='After (+UNet B)',    color='#DC143C', alpha=0.85)
ax.set_xticks(x); ax.set_xticklabels(labels, rotation=20, ha='right')
ax.set_ylim(0, 1.12); ax.set_ylabel('Score'); ax.legend()
for bar in list(b1) + list(b2):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.01,
            f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=8)
plt.tight_layout()
plt.savefig('/kaggle/working/before_after_unet.png', dpi=150, bbox_inches='tight')
plt.show()

Inference & Diagnostics
Cells 14–19: run after training is complete.

In [ ]:
# ==============================================================================
# CELL 27 — Inference Config (run after training is complete)
# ==============================================================================
import numpy as np
np.trapz = np.trapezoid

MODEL_PATH       = '/kaggle/working/runs/detect/stageC_unet_p2/weights/best.pt'
TEST_DIR         = '/kaggle/working/Thermal-H&C-1/test/images'
LABEL_DIR        = '/kaggle/working/Thermal-H&C-1/test/labels'
YAML_PATH        = '/kaggle/working/data_fixed.yaml'
OUTPUT_DIR       = '/kaggle/working/fadnet_unet_inference'
CONF             = 0.05
IOU              = 0.35
CLASS_NAMES      = ['Crack', 'Hotspot']
CLASS_COLORS_BGR = {0: (0, 0, 220), 1: (0, 140, 255)}

os.makedirs(OUTPUT_DIR, exist_ok=True)
infer_model = YOLO(MODEL_PATH)
test_images = sorted(glob.glob(os.path.join(TEST_DIR, '*.*')))
print(f'Model loaded ✓  |  Test images: {len(test_images)}')
print(f'GPU: {torch.cuda.get_device_name(0)}')


In [ ]:
# ==============================================================================
# CELL 28 — Single Image Inference + Annotated Visualization
# ==============================================================================
def infer_single(image_path, conf=CONF, iou=IOU, save=True):
    img_bgr = cv2.imread(image_path)
    t0 = time.perf_counter()
    results = infer_model.predict(image_path, conf=conf, iou=iou, verbose=False)[0]
    latency_ms = (time.perf_counter() - t0) * 1000
    annotated = img_bgr.copy(); counts = defaultdict(int)
    for box in results.boxes:
        cls_id = int(box.cls.item()); conf_v = box.conf.item()
        x1,y1,x2,y2 = map(int, box.xyxy[0].tolist())
        color = CLASS_COLORS_BGR[cls_id]; label = f'{CLASS_NAMES[cls_id]} {conf_v:.2f}'
        counts[CLASS_NAMES[cls_id]] += 1
        cv2.rectangle(annotated,(x1,y1),(x2,y2),color,2)
        (tw,th),_ = cv2.getTextSize(label,cv2.FONT_HERSHEY_SIMPLEX,0.55,1)
        cv2.rectangle(annotated,(x1,y1-th-6),(x1+tw+4,y1),color,-1)
        cv2.putText(annotated,label,(x1+2,y1-4),cv2.FONT_HERSHEY_SIMPLEX,0.55,(255,255,255),1,cv2.LINE_AA)
    for i,line in enumerate([
        f'FADNet+UNet | {Path(image_path).name}',
        f'Latency: {latency_ms:.1f} ms',
        f'Crack: {counts["Crack"]}',
        f'Hotspot: {counts["Hotspot"]}',
    ]):
        yp = 22+i*22
        cv2.putText(annotated,line,(10,yp),cv2.FONT_HERSHEY_SIMPLEX,0.55,(0,0,0),3,cv2.LINE_AA)
        cv2.putText(annotated,line,(10,yp),cv2.FONT_HERSHEY_SIMPLEX,0.55,(255,255,255),1,cv2.LINE_AA)
    if save:
        cv2.imwrite(os.path.join(OUTPUT_DIR,'single_'+Path(image_path).name), annotated)
    rgb = cv2.cvtColor(annotated,cv2.COLOR_BGR2RGB)
    plt.figure(figsize=(10,6)); plt.imshow(rgb); plt.axis('off')
    plt.title(f'FADNet+UNet  |  {latency_ms:.1f}ms  |  Crack:{counts["Crack"]}  Hotspot:{counts["Hotspot"]}')
    plt.legend(handles=[
        mpatches.Patch(color=(220/255,0,0),label='Crack'),
        mpatches.Patch(color=(1,140/255,0),label='Hotspot'),
    ],loc='upper right',fontsize=9)
    plt.tight_layout(); plt.show()
    return annotated, results

annotated_img, raw_result = infer_single(test_images[0])


In [ ]:
# ==============================================================================
# CELL 29 — Batch Inference on Entire Test Set
# ==============================================================================
BATCH_OUT = os.path.join(OUTPUT_DIR, 'batch_annotated')
os.makedirs(BATCH_OUT, exist_ok=True)
latencies = []; det_counts = defaultdict(list)

for img_path in tqdm(test_images, desc='Batch inference'):
    img_bgr = cv2.imread(img_path); annotated = img_bgr.copy()
    t0 = time.perf_counter()
    result = infer_model.predict(img_path, conf=CONF, iou=IOU, verbose=False)[0]
    latencies.append((time.perf_counter()-t0)*1000)
    counts = defaultdict(int)
    for box in result.boxes:
        cls_id=int(box.cls.item()); conf_v=box.conf.item()
        x1,y1,x2,y2=map(int,box.xyxy[0].tolist())
        color=CLASS_COLORS_BGR[cls_id]; label=f'{CLASS_NAMES[cls_id]} {conf_v:.2f}'
        counts[CLASS_NAMES[cls_id]]+=1
        cv2.rectangle(annotated,(x1,y1),(x2,y2),color,2)
        (tw,th),_=cv2.getTextSize(label,cv2.FONT_HERSHEY_SIMPLEX,0.5,1)
        cv2.rectangle(annotated,(x1,y1-th-5),(x1+tw+4,y1),color,-1)
        cv2.putText(annotated,label,(x1+2,y1-3),cv2.FONT_HERSHEY_SIMPLEX,0.5,(255,255,255),1,cv2.LINE_AA)
    for name in CLASS_NAMES: det_counts[name].append(counts[name])
    cv2.imwrite(os.path.join(BATCH_OUT, Path(img_path).name), annotated)

lat = np.array(latencies)
print(f'Batch done: {len(test_images)} images')
print(f'Latency  mean={lat.mean():.1f}ms  p50={np.percentile(lat,50):.1f}ms  '
      f'p95={np.percentile(lat,95):.1f}ms  max={lat.max():.1f}ms')
print(f'Avg dets/img  Crack={np.mean(det_counts["Crack"]):.2f}  '
      f'Hotspot={np.mean(det_counts["Hotspot"]):.2f}')


In [ ]:
# ==============================================================================
# CELL 30 — Full Per-Class Metrics + P/R/F1/AP Charts
# ==============================================================================
from pathlib import Path as _Path

metrics = infer_model.val(
    data=YAML_PATH, conf=CONF, iou=IOU, split='test',
    plots=True, save_dir=_Path(OUTPUT_DIR) / 'val_plots',
)

print('\n' + '='*60)
print('FADNet+UNet — Full Test-Split Diagnostics')
print('='*60)
print(f'\n[Overall]')
print(f'  mAP@0.5:      {metrics.box.map50:.4f}')
print(f'  mAP@0.5:0.95: {metrics.box.map:.4f}')
print(f'  Precision:    {metrics.box.mp:.4f}')
print(f'  Recall:       {metrics.box.mr:.4f}')
f1 = 2*metrics.box.mp*metrics.box.mr/(metrics.box.mp+metrics.box.mr+1e-9)
print(f'  F1:           {f1:.4f}')
print(f'\n[Per-Class]')
for i, name in enumerate(CLASS_NAMES):
    p=metrics.box.p[i]; r=metrics.box.r[i]; fi=2*p*r/(p+r+1e-9)
    print(f'  {name:<10} P={p:.4f}  R={r:.4f}  F1={fi:.4f}  '
          f'AP50={metrics.box.ap50[i]:.4f}  AP50-95={metrics.box.ap[i]:.4f}')
spd=metrics.speed; total_ms=sum(spd.values())
print(f'\n[Speed]  infer={spd["inference"]:.2f}ms  total={total_ms:.2f}ms  FPS={1000/total_ms:.1f}')
print('='*60)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('FADNet+UNet — AP & P/R/F1 Summary', fontsize=13, fontweight='bold')
x = np.arange(len(CLASS_NAMES))
b1=axes[0].bar(x-0.2,[metrics.box.ap50[i] for i in range(len(CLASS_NAMES))],
               0.35,label='AP@0.5',color=['#DC143C','#FF8C00'])
b2=axes[0].bar(x+0.2,[metrics.box.ap[i]   for i in range(len(CLASS_NAMES))],
               0.35,label='AP@0.5:0.95',color=['#FF6666','#FFB347'])
axes[0].set_xticks(x); axes[0].set_xticklabels(CLASS_NAMES)
axes[0].set_ylim(0,1.1); axes[0].legend()
for bar in list(b1)+list(b2):
    axes[0].text(bar.get_x()+bar.get_width()/2,bar.get_height()+0.01,
                 f'{bar.get_height():.3f}',ha='center',va='bottom',fontsize=9)
p_v=[metrics.box.p[i] for i in range(len(CLASS_NAMES))]
r_v=[metrics.box.r[i] for i in range(len(CLASS_NAMES))]
f1_v=[2*p*r/(p+r+1e-9) for p,r in zip(p_v,r_v)]
for vals,offset,color,label in zip([p_v,r_v,f1_v],[-0.25,0,0.25],
                                    ['#4682B4','#228B22','#8B008B'],
                                    ['Precision','Recall','F1']):
    bars=axes[1].bar(x+offset,vals,0.22,label=label,color=color)
    for bar,v in zip(bars,vals):
        axes[1].text(bar.get_x()+bar.get_width()/2,bar.get_height()+0.01,
                     f'{v:.3f}',ha='center',va='bottom',fontsize=8)
axes[1].set_xticks(x); axes[1].set_xticklabels(CLASS_NAMES)
axes[1].set_ylim(0,1.1); axes[1].legend()
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR,'metrics_summary.png'),dpi=150,bbox_inches='tight')
plt.show()


In [ ]:
# ==============================================================================
# CELL 31 — Confidence Distribution + Box Size + Detection Count
# ==============================================================================
conf_scores=defaultdict(list); box_areas=defaultdict(list); boxes_per_image=defaultdict(list)

for img_path in tqdm(test_images, desc='Confidence analysis'):
    result=infer_model.predict(img_path,conf=0.01,iou=IOU,verbose=False)[0]
    H,W=result.orig_shape; counts=defaultdict(int)
    for box in result.boxes:
        cls_id=int(box.cls.item()); cv_val=box.conf.item()
        x1,y1,x2,y2=box.xyxy[0].tolist(); name=CLASS_NAMES[cls_id]
        conf_scores[name].append(cv_val)
        box_areas[name].append(((x2-x1)*(y2-y1))/(H*W))
        counts[name]+=1
    for name in CLASS_NAMES: boxes_per_image[name].append(counts[name])

colors_hex={'Crack':'#DC143C','Hotspot':'#FF8C00'}
fig,axes=plt.subplots(2,2,figsize=(14,9))
fig.suptitle('FADNet+UNet — Confidence & Detection Analysis',fontsize=13,fontweight='bold')
for name in CLASS_NAMES:
    axes[0,0].hist(conf_scores[name],bins=30,alpha=0.6,label=name,
                   color=colors_hex[name],edgecolor='white')
axes[0,0].axvline(CONF,color='black',linestyle='--',linewidth=1.2,label=f'thr={CONF}')
axes[0,0].set_xlabel('Confidence');axes[0,0].set_ylabel('Count');axes[0,0].legend()
axes[0,0].set_title('Confidence Distribution (conf=0.01)')
for name in CLASS_NAMES:
    s=np.sort(conf_scores[name])
    axes[0,1].plot(s,np.arange(1,len(s)+1)/len(s),label=name,color=colors_hex[name],linewidth=2)
axes[0,1].axvline(CONF,color='black',linestyle='--',linewidth=1.2)
axes[0,1].set_xlabel('Confidence');axes[0,1].set_ylabel('CDF');axes[0,1].legend()
axes[0,1].set_title('Confidence CDF')
for name in CLASS_NAMES:
    axes[1,0].hist(np.array(box_areas[name])*100,bins=25,alpha=0.6,
                   label=name,color=colors_hex[name],edgecolor='white')
axes[1,0].set_xlabel('Box Area (% of image)');axes[1,0].set_ylabel('Count')
axes[1,0].set_title('Detected Box Size Distribution');axes[1,0].legend()
bp=axes[1,1].boxplot([boxes_per_image[n] for n in CLASS_NAMES],
                      labels=CLASS_NAMES,patch_artist=True,
                      medianprops=dict(color='black',linewidth=2))
for patch,name in zip(bp['boxes'],CLASS_NAMES):
    patch.set_facecolor(colors_hex[name]);patch.set_alpha(0.6)
axes[1,1].set_ylabel('Detections per Image');axes[1,1].set_title('Detection Count per Image')
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR,'confidence_analysis.png'),dpi=150,bbox_inches='tight')
plt.show()


In [ ]:
# ==============================================================================
# CELL 32 — FP / FN Diagnosis: Worst Misses & False Alarms
# ==============================================================================
def load_gt(label_path,img_w,img_h):
    boxes=[]
    if not os.path.exists(label_path): return boxes
    with open(label_path) as f:
        for line in f:
            p=line.strip().split()
            if len(p)<5: continue
            cls_id=int(p[0]); cx,cy,bw,bh=map(float,p[1:5])
            x1=(cx-bw/2)*img_w;y1=(cy-bh/2)*img_h
            x2=(cx+bw/2)*img_w;y2=(cy+bh/2)*img_h
            boxes.append({'cls':cls_id,'box':[x1,y1,x2,y2]})
    return boxes

def iou_calc(b1,b2):
    xi1=max(b1[0],b2[0]);yi1=max(b1[1],b2[1])
    xi2=min(b1[2],b2[2]);yi2=min(b1[3],b2[3])
    inter=max(0,xi2-xi1)*max(0,yi2-yi1)
    return inter/((b1[2]-b1[0])*(b1[3]-b1[1])+(b2[2]-b2[0])*(b2[3]-b2[1])-inter+1e-9)

IOU_MATCH=0.35
class_tp=defaultdict(int);class_fp=defaultdict(int);class_fn=defaultdict(int)
fp_images=[];fn_images=[]

for img_path in tqdm(test_images,desc='FP/FN diagnosis'):
    stem=Path(img_path).stem;img_bgr=cv2.imread(img_path);H,W=img_bgr.shape[:2]
    gt=load_gt(os.path.join(LABEL_DIR,stem+'.txt'),W,H)
    result=infer_model.predict(img_path,conf=CONF,iou=IOU,verbose=False)[0]
    preds=[{'cls':int(b.cls.item()),'conf':b.conf.item(),'box':b.xyxy[0].tolist()} for b in result.boxes]
    matched_gt=set();matched_pred=set()
    for pi,pred in enumerate(preds):
        best_iou,best_gi=0,-1
        for gi,g in enumerate(gt):
            if g['cls']!=pred['cls'] or gi in matched_gt: continue
            v=iou_calc(pred['box'],g['box'])
            if v>best_iou: best_iou,best_gi=v,gi
        if best_iou>=IOU_MATCH:
            matched_gt.add(best_gi);matched_pred.add(pi)
            class_tp[CLASS_NAMES[pred['cls']]]+=1
    fp_list=[preds[i] for i in range(len(preds)) if i not in matched_pred]
    fn_list=[gt[i] for i in range(len(gt)) if i not in matched_gt]
    for fp in fp_list: class_fp[CLASS_NAMES[fp['cls']]]+=1
    for fn in fn_list: class_fn[CLASS_NAMES[fn['cls']]]+=1
    if fp_list: fp_images.append((img_path,len(fp_list),fp_list))
    if fn_list: fn_images.append((img_path,len(fn_list),fn_list))

print('\n'+'='*60)
print('FP / FN / TP Breakdown')
print('='*60)
for name in CLASS_NAMES:
    tp=class_tp[name];fp_c=class_fp[name];fn_c=class_fn[name]
    prec=tp/(tp+fp_c+1e-9);rec=tp/(tp+fn_c+1e-9)
    print(f'  {name:<10}  TP={tp:>4}  FP={fp_c:>4}  FN={fn_c:>4}  Prec={prec:.3f}  Rec={rec:.3f}')
print(f'  Images w/ FP: {len(fp_images)}/{len(test_images)}')
print(f'  Images w/ FN: {len(fn_images)}/{len(test_images)}')
print('='*60)

def show_worst(cases,n,title,box_color,lbl_prefix):
    cases=sorted(cases,key=lambda x:x[1],reverse=True)
    n_show=min(n,len(cases))
    if n_show==0: return
    fig,axes=plt.subplots(1,n_show,figsize=(5*n_show,5))
    if n_show==1: axes=[axes]
    fig.suptitle(title,fontsize=12,fontweight='bold')
    for ax,(ip,count,boxes) in zip(axes,cases[:n_show]):
        img=cv2.cvtColor(cv2.imread(ip),cv2.COLOR_BGR2RGB)
        for b in boxes:
            x1,y1,x2,y2=map(int,b['box'])
            cv2.rectangle(img,(x1,y1),(x2,y2),box_color,2)
            lbl=f'{lbl_prefix}:{CLASS_NAMES[b["cls"]]}'+( f' {b["conf"]:.2f}' if 'conf' in b else '')
            cv2.putText(img,lbl,(x1,max(y1-4,10)),cv2.FONT_HERSHEY_SIMPLEX,0.5,box_color,1)
        ax.imshow(img);ax.set_title(f'{Path(ip).name}\ncount={count}',fontsize=9);ax.axis('off')
    plt.tight_layout()
    fname=title.replace(' ','_').replace('/','').lower()[:30]+'.png'
    plt.savefig(os.path.join(OUTPUT_DIR,fname),dpi=150,bbox_inches='tight')
    plt.show()

show_worst(fn_images,4,'Worst False Negatives (missed GT)',(0,200,0),'MISSED')
show_worst(fp_images,4,'Worst False Positives (wrong pred)',(30,30,220),'FP')
print('FP/FN plots saved ✓')


In [ ]:
# ==============================================================================
# FINAL CELL — Save & Download Best Model from Each Stage
# Copies the three key checkpoints to /kaggle/working/ with clear names,
# then pushes them to a versioned Kaggle dataset for persistent storage.
# ==============================================================================
import os, shutil, json, subprocess
from pathlib import Path

# ── checkpoint map: label → source path ────────────────────────────────────
CKPTS = {
    'fadnet_yolo_best.pt':    '/kaggle/working/runs/detect/stageC_aug_v2_p2/weights/best.pt',
    'fadnet_unet_best.pth':   '/kaggle/working/unet_stageb_best.pth',
    'fadnet_finetune_best.pt': '/kaggle/working/runs/detect/stageC_unet_p2/weights/best.pt',
}

SAVE_DIR = '/kaggle/working/fadnet_final_checkpoints'
os.makedirs(SAVE_DIR, exist_ok=True)

print('Copying checkpoints ...')
missing = []
for dst_name, src_path in CKPTS.items():
    dst = os.path.join(SAVE_DIR, dst_name)
    if os.path.exists(src_path):
        shutil.copy2(src_path, dst)
        size_mb = os.path.getsize(dst) / 1e6
        print(f'  ✓  {dst_name:<35}  {size_mb:.1f} MB')
    else:
        print(f'  ✗  MISSING: {src_path}')
        missing.append(src_path)

if missing:
    print(f'\n⚠  {len(missing)} checkpoint(s) not found — check training completed for all stages.')

# ── push to Kaggle dataset ─────────────────────────────────────────────────
DATASET_SLUG = 'fadnet-final-models'   # change if needed

meta = {
    "title": "FADNet Final Models",
    "id": f"vishokbadri/{DATASET_SLUG}",
    "licenses": [{"name": "CC0-1.0"}]
}
meta_path = os.path.join(SAVE_DIR, 'dataset-metadata.json')
with open(meta_path, 'w') as f:
    json.dump(meta, f)

print('\nPushing to Kaggle dataset ...')
try:
    result = subprocess.run(
        ['kaggle', 'datasets', 'version', '-p', SAVE_DIR,
         '-m', 'FADNet checkpoints: YOLO StageC + UNet + FineTune',
         '--dir-mode', 'zip'],
        capture_output=True, text=True, timeout=300
    )
    if result.returncode == 0:
        print(result.stdout)
        print(f'✓  Dataset updated: https://www.kaggle.com/datasets/vishokbadri/{DATASET_SLUG}')
    else:
        # Dataset may not exist yet — create it
        create_result = subprocess.run(
            ['kaggle', 'datasets', 'create', '-p', SAVE_DIR, '--dir-mode', 'zip'],
            capture_output=True, text=True, timeout=300
        )
        print(create_result.stdout or create_result.stderr)
        if create_result.returncode == 0:
            print(f'✓  Dataset created: https://www.kaggle.com/datasets/vishokbadri/{DATASET_SLUG}')
        else:
            print('Kaggle push failed. Files are still available as notebook output in:')
            print(f'  {SAVE_DIR}')
except Exception as e:
    print(f'Kaggle CLI error: {e}')
    print('Files saved as notebook output in:', SAVE_DIR)

print('\n── Checkpoint Summary ──')
for f in sorted(os.listdir(SAVE_DIR)):
    if not f.endswith('.json'):
        fp = os.path.join(SAVE_DIR, f)
        print(f'  {f:<40}  {os.path.getsize(fp)/1e6:.1f} MB')
